# T34 - 债券新发行

## 第7章：部署与监控

---

### 本章概述

本章实现债券新发行系统的部署和监控功能，包括：
1. 定时任务配置
2. 日志管理
3. 错误告警
4. 运维脚本

### 导入依赖

In [ ]:
# 标准库
import os
import sys
import datetime
from datetime import datetime, timedelta
import logging
import json
import traceback
import warnings
from pathlib import Path

# 数据处理
import pandas as pd

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入成功")

### 日志配置

In [ ]:
def setup_logging(log_dir: str = 'logs', log_level: str = 'INFO'):
    """
    配置日志系统
    
    Args:
        log_dir: 日志目录
        log_level: 日志级别
    """
    # 创建日志目录
    log_path = Path(log_dir)
    log_path.mkdir(parents=True, exist_ok=True)
    
    # 日志文件名
    log_file = log_path / f"bond_new_issue_{datetime.now().strftime('%Y%m%d')}.log"
    
    # 配置日志
    logging.basicConfig(
        level=getattr(logging, log_level),
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler(sys.stdout)
        ]
    )
    
    return logging.getLogger('BondNewIssue')


logger = setup_logging()
logger.info("日志系统初始化完成")

### 任务调度器

In [ ]:
class TaskScheduler:
    """任务调度器"""
    
    def __init__(self, logger=None):
        self.logger = logger or logging.getLogger()
        self.tasks = []
        self.results = []
    
    def add_task(self, name: str, func: callable, schedule_time: str = None, **kwargs):
        """
        添加任务
        
        Args:
            name: 任务名称
            func: 任务函数
            schedule_time: 计划执行时间 (HH:MM格式)
            **kwargs: 任务参数
        """
        self.tasks.append({
            'name': name,
            'func': func,
            'schedule_time': schedule_time,
            'kwargs': kwargs
        })
        self.logger.info(f"添加任务: {name}")
    
    def run_task(self, task: dict) -> dict:
        """
        执行单个任务
        
        Args:
            task: 任务配置
            
        Returns:
            执行结果
        """
        result = {
            'name': task['name'],
            'start_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'status': 'UNKNOWN',
            'message': '',
            'duration': 0
        }
        
        start = datetime.now()
        
        try:
            self.logger.info(f"开始执行任务: {task['name']}")
            task_result = task['func'](**task.get('kwargs', {}))
            
            result['status'] = 'SUCCESS'
            result['message'] = '任务执行成功'
            result['result'] = task_result
            
            self.logger.info(f"任务执行成功: {task['name']}")
            
        except Exception as e:
            result['status'] = 'FAILED'
            result['message'] = str(e)
            result['traceback'] = traceback.format_exc()
            
            self.logger.error(f"任务执行失败: {task['name']}, 错误: {e}")
            self.logger.error(traceback.format_exc())
        
        finally:
            result['end_time'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            result['duration'] = (datetime.now() - start).total_seconds()
        
        return result
    
    def run_all(self) -> list:
        """
        执行所有任务
        
        Returns:
            所有任务的执行结果
        """
        self.logger.info(f"开始执行 {len(self.tasks)} 个任务")
        
        self.results = []
        
        for task in self.tasks:
            result = self.run_task(task)
            self.results.append(result)
        
        # 统计
        success_count = sum(1 for r in self.results if r['status'] == 'SUCCESS')
        failed_count = sum(1 for r in self.results if r['status'] == 'FAILED')
        
        self.logger.info(f"任务执行完成: 成功 {success_count}, 失败 {failed_count}")
        
        return self.results
    
    def get_summary(self) -> dict:
        """获取执行摘要"""
        if not self.results:
            return {'status': 'no_results'}
        
        success = [r for r in self.results if r['status'] == 'SUCCESS']
        failed = [r for r in self.results if r['status'] == 'FAILED']
        
        total_duration = sum(r['duration'] for r in self.results)
        
        return {
            'total_tasks': len(self.results),
            'success_count': len(success),
            'failed_count': len(failed),
            'total_duration': round(total_duration, 2),
            'failed_tasks': [r['name'] for r in failed]
        }


print("TaskScheduler类定义完成")

### 告警通知

In [ ]:
class AlertManager:
    """告警管理器"""
    
    def __nit__(self, logger=None):
        self.logger = logger or logging.getLogger()
        self.alerts = []
    
    def send_alert(self, level: str, title: str, message: str, details: dict = None):
        """
        发送告警
        
        Args:
            level: 告警级别 (INFO, WARNING, ERROR, CRITICAL)
            title: 告警标题
            message: 告警消息
            details: 详细信息
        """
        alert = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'level': level,
            'title': title,
            'message': message,
            'details': details or {}
        }
        
        self.alerts.append(alert)
        
        # 记录日志
        log_method = getattr(self.logger, level.lower(), self.logger.info)
        log_method(f"[{level}] {title}: {message}")
        
        # 这里可以扩展为发送邮件、钉钉、企业微信等通知
        # self._send_email(alert)
        # self._send_dingtalk(alert)
    
    def check_result_and_alert(self, result: dict):
        """
        根据执行结果发送告警
        
        Args:
            result: 任务执行结果
        """
        if result['status'] == 'FAILED':
            self.send_alert(
                level='ERROR',
                title=f"任务失败: {result['name']}",
                message=result['message'],
                details={'traceback': result.get('traceback', '')}
            )
        elif result['status'] == 'SUCCESS':
            self.send_alert(
                level='INFO',
                title=f"任务成功: {result['name']}",
                message=f"执行耗时: {result['duration']:.2f}秒"
            )
    
    def get_alerts(self, level: str = None) -> list:
        """
        获取告警列表
        
        Args:
            level: 过滤级别
            
        Returns:
            告警列表
        """
        if level:
            return [a for a in self.alerts if a['level'] == level]
        return self.alerts


print("AlertManager类定义完成")

### 主程序入口

In [ ]:
def main():
    """
    主程序入口
    """
    logger = setup_logging()
    logger.info("="*60)
    logger.info("债券新发行系统启动")
    logger.info("="*60)
    
    # 创建调度器和告警管理器
    scheduler = TaskScheduler(logger)
    alert_manager = AlertManager(logger)
    
    # 定义任务函数（占位）
    def collect_maturity_data():
        """采集债券到期数据"""
        # from 第5章 import BondMaturityCollector
        # collector = BondMaturityCollector(engine)
        # return collector.collect(days_ahead=7)
        logger.info("执行债券到期数据采集")
        return {'success': True, 'records': 0}
    
    def collect_issue_data():
        """采集债券新发行数据"""
        # from 第5章 import BondIssueCollector
        # collector = BondIssueCollector(engine)
        # return collector.collect(days_start=1, days_end=30)
        logger.info("执行债券新发行数据采集")
        return {'success': True, 'records': 0}
    
    def check_data_quality():
        """检查数据质量"""
        # from 第6章 import DataQualityChecker
        # checker = DataQualityChecker(engine)
        # return checker.run_all_checks()
        logger.info("执行数据质量检查")
        return {'success': True}
    
    # 添加任务
    scheduler.add_task('债券到期数据采集', collect_maturity_data)
    scheduler.add_task('债券新发行数据采集', collect_issue_data)
    scheduler.add_task('数据质量检查', check_data_quality)
    
    # 执行任务
    results = scheduler.run_all()
    
    # 检查结果并发送告警
    for result in results:
        alert_manager.check_result_and_alert(result)
    
    # 获取摘要
    summary = scheduler.get_summary()
    
    logger.info("="*60)
    logger.info(f"执行摘要: {summary}")
    logger.info("="*60)
    
    return summary


print("主程序定义完成")

### 定时任务配置（Windows）

In [ ]:
# Windows 任务计划程序配置示例
#
# 创建每天运行的批处理文件 run_daily.bat:
#
# @echo off
# chcp 65001 > nul
# cd /d "g:\我的云端硬盘\workspace\projects\github\work_station\重构\T34-债券新发行"
# python main.py
#
# 然后在Windows任务计划程序中:
# 1. 创建基本任务
# 2. 设置触发器: 每天 16:00
# 3. 设置操作: 启动程序 run_daily.bat

print("Windows定时任务配置说明已提供")

### 定时任务配置（Linux/Cron）

In [ ]:
# Linux Crontab 配置示例
#
# 编辑crontab:
# crontab -e
#
# 添加以下行（每天16:00执行）:
# 0 16 * * * cd /path/to/T34-债券新发行 && python main.py >> logs/cron.log 2>&1
#

print("Linux定时任务配置说明已提供")

### 运行示例

In [ ]:
# 执行主程序
if __name__ == "__main__":
    summary = main()
    print("\n执行摘要:")
    print(json.dumps(summary, indent=2, ensure_ascii=False))

---

### 本章小结

**已实现功能**:
- 日志系统配置
- `TaskScheduler`: 任务调度器
- `AlertManager`: 告警管理器
- 主程序入口
- 定时任务配置说明

**部署建议**:
1. 配置环境变量（数据库密码、API凭证）
2. 设置定时任务（建议每天16:00执行）
3. 配置告警通知（邮件/钉钉/企业微信）
4. 定期检查日志和数据质量

---

### 项目总结

**T34 债券新发行系统** 已完成重构，包含以下模块：

| 章节 | 内容 |
|------|------|
| 第1章 | 项目概述与数据字典 |
| 第2章 | 环境配置与初始化 |
| 第3章 | 数据采集与ETL |
| 第4章 | 数据处理模块 |
| 第5章 | 核心逻辑实现 |
| 第6章 | 测试与验证 |
| 第7章 | 部署与监控 |

**重构亮点**:
- 使用环境变量管理敏感配置
- 面向对象的设计，便于维护和扩展
- 完善的日志和告警机制
- 数据质量检查功能
- 定时任务调度支持